In [68]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

In [69]:
def clean_text(text):
    text = str(text)
    
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [70]:
def load_imdb(path):
    df = pd.read_csv(path)

    df = df[['review', 'sentiment']]

    df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
    df['text'] = df['review'].apply(clean_text)

    df = df[['text', 'label']]

    return df

In [92]:
def load_rotten_tomatoes(path):
    df = pd.read_csv(path)

    df = df[['review_content', 'review_type']].dropna()

    df['label'] = df['review_type'].map({
        'fresh': 1,
        'rotten': 0,
        'Fresh': 1,
        'Rotten': 0
    })

    df = df.dropna(subset=['label'])

    df['text'] = df['review_content'].apply(clean_text)

    df = df[['text', 'label']].copy()

    df_0 = df[df['label'] == 0]
    df_1 = df[df['label'] == 1]

    n = 25_000

    df_0 = df_0.sample(n=min(len(df_0), n), random_state=42)
    df_1 = df_1.sample(n=min(len(df_1), n), random_state=42)

    df_balanced = pd.concat([df_0, df_1], axis=0)

    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    # DEBUG (to jest ważne — od razu widzisz czy działa)
    print("\nFinal distribution:")
    print(df_balanced['label'].value_counts())
    print(df_balanced.head())

    return df_balanced  

In [72]:
def load_amazon(path):
    df = pd.read_csv(
        path,
        engine='python',
        on_bad_lines='skip'
    )

    df = df[['Review Text', 'Rating']].dropna()

    df['Rating'] = df['Rating'].astype(str).str.extract(r'(\d+)')
    df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

    df = df.dropna(subset=['Rating'])

    df['Rating'] = df['Rating'].astype(int)

    df['label'] = df['Rating'].apply(lambda x: 1 if x >= 3 else 0)

    df['text'] = df['Review Text'].apply(clean_text)

    return df[['text', 'label']]

In [73]:
def load_yelp(path):
    df = pd.read_csv(path)

    df = df[['text', 'stars']].dropna()

    df['label'] = df['stars'].apply(lambda x: 1 if x >= 3 else 0)

    df['text'] = df['text'].apply(clean_text)

    return df[['text', 'label']]

In [90]:
def prepare_all_datasets(paths, balance=True):
    datasets = {}

    datasets['imdb'] = load_imdb(paths['imdb'])
    datasets['rotten'] = load_rotten_tomatoes(paths['rotten'])
    datasets['amazon'] = load_amazon(paths['amazon'])
    datasets['yelp'] = load_yelp(paths['yelp'])

    return datasets

In [93]:
paths = {
    'imdb': '../datasets/raw/imdb.csv',
    'rotten': '../datasets/raw/rotten_tomatoes.csv',
    'amazon': '../datasets/raw/amazon.csv',
    'yelp': '../datasets/raw/yelp.csv'
}

datasets = prepare_all_datasets(paths)

datasets


Final distribution:
label
1    25000
0    25000
Name: count, dtype: int64
                                                text  label
0  a few daring antiheroes who have nothing to lo...      1
1  if anything it made me truly appreciate the am...      0
2  i found something questionable in its forced a...      0
3  cruising into cinemas with worse timing than a...      0
4                    a delicious feast of filmmaking      1


{'imdb':                                                     text  label
 0      one of the other reviewers has mentioned that ...      1
 1      a wonderful little production the filming tech...      1
 2      i thought this was a wonderful way to spend ti...      1
 3      basically theres a family where a little boy j...      0
 4      petter matteis love in the time of money is a ...      1
 ...                                                  ...    ...
 49995  i thought this movie did a down right good job...      1
 49996  bad plot bad dialogue bad acting idiotic direc...      0
 49997  i am a catholic taught in parochial elementary...      0
 49998  im going to have to disagree with the previous...      0
 49999  no one expects the star trek movies to be high...      0
 
 [50000 rows x 2 columns],
 'rotten':                                                     text  label
 0      a few daring antiheroes who have nothing to lo...      1
 1      if anything it made me truly apprec

In [94]:
import os
def save_datasets(datasets, base_path='../datasets/processed'):
    os.makedirs(base_path, exist_ok=True)

    for name, df in datasets.items():
        save_path = f"{base_path}/{name}.csv"
        df.to_csv(save_path, index=False)
        print(f"Saved: {save_path} ({len(df)} rows)")

In [95]:
save_datasets(datasets)

Saved: ../datasets/processed/imdb.csv (50000 rows)
Saved: ../datasets/processed/rotten.csv (50000 rows)
Saved: ../datasets/processed/amazon.csv (21055 rows)
Saved: ../datasets/processed/yelp.csv (10000 rows)


In [96]:
def count_labels(datasets):
    for name, df in datasets.items():
        counts = df['label'].value_counts().sort_index()

        print(f"\n=== {name} ===")
        print(f"Class 0 (negative): {counts.get(0, 0)}")
        print(f"Class 1 (positive): {counts.get(1, 0)}")
        print(f"Total: {len(df)}")

In [97]:
count_labels(datasets)


=== imdb ===
Class 0 (negative): 25000
Class 1 (positive): 25000
Total: 50000

=== rotten ===
Class 0 (negative): 25000
Class 1 (positive): 25000
Total: 50000

=== amazon ===
Class 0 (negative): 14350
Class 1 (positive): 6705
Total: 21055

=== yelp ===
Class 0 (negative): 1676
Class 1 (positive): 8324
Total: 10000
